# Import Libraries and Retrieve Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from xgboost import XGBRegressor
from pathlib import Path
import json
from sklearn.linear_model import LinearRegression
import joblib

In [2]:
#Pull data from raw folder and compile together into one df

full_data = []
#Extract all files within raw folder and below
for filepath in Path("drive/MyDrive/raw").rglob("*.json"):
  with filepath.open("r", encoding="utf-8") as file:
    response_data = json.load(file)


    #Extract the AirQualityData from the response and get the Data section for processing
    air_quality_data = response_data.get("AirQualityData", {})
    data = air_quality_data.get("Data", [])

    #Makes sure that data is always a list, even if there's only one record
    if isinstance(data, dict):
      data = [data]

    #For each record in the data, get the relevant columns and append to records.
    for record in data:
      full_data.append({"site_code": air_quality_data.get("@SiteCode"),
                        "timestamp": record.get("@MeasurementDateGMT"),
                        "measurement": record.get("@Value")})

#Create the dataframe
full_df = pd.DataFrame(full_data)

full_df["timestamp"] = pd.to_datetime(full_df["timestamp"])
full_df["measurement"] = pd.to_numeric(full_df["measurement"], errors="coerce")

full_df["date"] = full_df["timestamp"].dt.normalize()
full_df["available"] = full_df["measurement"].notna()

In [3]:
display(full_df.head(50))

,site_code,timestamp,measurement,date,available
0,BG1,2026-05-01 00:00:00,NaN,2026-05-01,False
1,BG1,2026-05-01 01:00:00,NaN,2026-05-01,False
2,BG1,2026-05-01 02:00:00,NaN,2026-05-01,False
3,BG1,2026-05-01 03:00:00,NaN,2026-05-01,False
4,BG1,2026-05-01 04:00:00,NaN,2026-05-01,False
5,BG1,2026-05-01 05:00:00,NaN,2026-05-01,False
6,BG1,2026-05-01 06:00:00,NaN,2026-05-01,False
7,BG1,2026-05-01 07:00:00,NaN,2026-05-01,False
8,BG1,2026-05-01 08:00:00,NaN,2026-05-01,False
9,BG1,2026-05-01 09:00:00,NaN,2026-05-01,False


In [4]:
print(full_df.shape)

(1755072, 5)


# Data Preparation: Transform and Engineer Features

In [5]:
#From full_df group rows so its one site per day.
#With the total expected measurements across each site per day as well as the available measurements
df = (full_df.groupby(["site_code", "date"], as_index=False).agg(site_expected_daily=("measurement", "size"),
                                                                 site_available_daily=("available", "sum")))

#Site reliability daily column is the percentage of available measurements out of expected measurements for each site on that day
df["site_reliability_daily"] = (df["site_available_daily"] / df["site_expected_daily"]) * 100

#Sort in order
df = df.sort_values(["site_code", "date"]).reset_index(drop=True)

#site_reliability_lag_1 column: The site's reliability 1 day ago.
df["site_reliability_lag_1"] = df.groupby("site_code")["site_reliability_daily"].shift(1)

#site_reliability_lag_2 column: The site's reliability 2 days ago.
df["site_reliability_lag_2"] = df.groupby("site_code")["site_reliability_daily"].shift(2)

#site_reliability_lag_3 column: The site's reliability 7 days ago.
df["site_reliability_lag_3"] = df.groupby("site_code")["site_reliability_daily"].shift(7)

#site_reliability_mean_7 column: The average daily reliability over the previous 7 days.
df["site_reliability_rolling_mean_7"] = df.groupby("site_code")["site_reliability_daily"].transform(lambda s: s.shift(1).rolling(window=7, min_periods=7).mean())

#Day of the week column
df["day_of_the_week"] = (df["date"] + pd.Timedelta(days=2)).dt.dayofweek

#site_expected_daily_lag_1 column: The total number of measurements expected from that site yesterday.
df["site_expected_daily_lag_1"] = (df.groupby("site_code")["site_expected_daily"].shift(1))

#site_available_daily_lag_1 column: The number of measurements that actually had values available yesterday.
df["site_available_daily_lag_1"] = (df.groupby("site_code")["site_available_daily"].shift(1))

#target_forecast_day column: The site's reliability 2 days ahead (the value to predict).
df["target_forecast_day"] = (df.groupby("site_code")["site_reliability_daily"].shift(-2))

#target_forecast_week column: The site's average reliability across seven days.
df["target_forecast_week"] = df.groupby("site_code")["site_reliability_daily"].transform(lambda s: s.shift(-7).rolling(window=7, min_periods=7).mean())


In [6]:
display(df.iloc[1500:1510])

,site_code,date,site_expected_daily,site_available_daily,site_reliability_daily,site_reliability_lag_1,site_reliability_lag_2,site_reliability_lag_3,site_reliability_rolling_mean_7,day_of_the_week,site_expected_daily_lag_1,site_available_daily_lag_1,target_forecast_day,target_forecast_week
1500,BT4,2026-07-06,144,77,53.472222,48.611111,50.000000,45.138889,48.611111,2,144.0,70.0,49.305556,47.519841
1501,BT4,2026-07-07,144,67,46.527778,53.472222,48.611111,49.305556,49.801587,3,144.0,77.0,50.000000,45.535714
1502,BT4,2026-07-08,144,71,49.305556,46.527778,53.472222,47.916667,49.404762,4,144.0,67.0,50.000000,44.146825
1503,BT4,2026-07-09,144,72,50.000000,49.305556,46.527778,49.305556,49.603175,5,144.0,71.0,50.000000,44.146825
1504,BT4,2026-07-10,144,72,50.000000,50.000000,49.305556,50.000000,49.702381,6,144.0,72.0,47.916667,44.146825
1505,BT4,2026-07-11,144,72,50.000000,50.000000,50.000000,50.000000,49.702381,0,144.0,72.0,38.888889,44.047619
1506,BT4,2026-07-12,144,69,47.916667,50.000000,50.000000,48.611111,49.702381,1,144.0,72.0,32.638889,44.146825
1507,BT4,2026-07-13,144,56,38.888889,47.916667,50.000000,53.472222,49.603175,2,144.0,69.0,39.583333,45.634921
1508,BT4,2026-07-14,144,47,32.638889,38.888889,47.916667,46.527778,47.519841,3,144.0,56.0,50.000000,48.115079
1509,BT4,2026-07-15,144,57,39.583333,32.638889,38.888889,49.305556,45.535714,4,144.0,47.0,50.000000,49.603175


In [7]:
print(df.shape)

(19008, 14)


# Model Training: Tomorrow's Reliability Forecast

In [8]:
#XGBOOST: Tommorrow's Forecast

features = ["site_code",
            "day_of_the_week",
            "site_reliability_daily",
            "site_reliability_lag_1",
            "site_reliability_lag_2",
            "site_reliability_lag_3",
            "site_reliability_rolling_mean_7",
            "site_expected_daily_lag_1",
            "site_available_daily_lag_1"]

target = "target_forecast_day"

#Drop any rows with empty target and features. This removes any rows that can't retrieve historical or future values.
model_df = df.dropna(subset= features + [target]).copy()

#One-hot encode site_code + features.
X = pd.get_dummies(model_df[features], columns=["site_code"], dtype=int)
y = model_df[target]

In [9]:
print(model_df.shape)

(18360, 14)


In [10]:
#Split into train and test sets

#Get the list of dates
dates = sorted(model_df["date"].unique())

#Find the 80% split point
split_index = int(len(dates) * 0.8)
#Get the date at that position
split_date = dates[split_index]

#Compares split_date with model_df dates and determines if True or False
train_mask = model_df["date"] < split_date
test_mask = model_df["date"] >= split_date

#Selects only rows where train and test mask is true for X_train and X_test
X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

#Selects only rows where train and test mask is true for y_train and y_test
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Split date:", split_date)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Split date: 2026-08-07 00:00:00
X_train: (14688, 80)
X_test: (3672, 80)
y_train: (14688,)
y_test: (3672,)


In [11]:
#Tune Hyperparams
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    objective="reg:squarederror",
    random_state=42)

In [12]:
#Train
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [13]:
#Test
predictions = model.predict(X_test)

In [14]:
#Evaluate
mae = mean_absolute_error(y_test, predictions)
rmse = root_mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

MAE: 5.11
RMSE: 13.29
R²: 0.8927


# Model Training: Next Seven Days' Average Reliability Forecast

In [15]:
#XGBOOST: Average for the next seven days

second_features = ["site_code",
                   "day_of_the_week",
                   "site_reliability_daily",
                   "site_reliability_lag_1",
                   "site_reliability_lag_2",
                   "site_reliability_lag_3",
                   "site_reliability_rolling_mean_7",
                   "site_expected_daily_lag_1",
                   "site_available_daily_lag_1"]

second_target = "target_forecast_week"

#Drop any rows with empty target and features. This removes any rows that can't retrieve historical or future values.
second_model_df = df.dropna(subset= second_features + [second_target]).copy()

#One-hot encode site_code + features.
second_X = pd.get_dummies(second_model_df[second_features], columns=["site_code"], dtype=int)
second_y = second_model_df[second_target]

In [16]:
print(second_model_df.shape)

(18000, 14)


In [17]:
#Split into train and test sets

#Get the list of dates
second_dates = sorted(second_model_df["date"].unique())

#Find the 80% split point
second_split_index = int(len(second_dates) * 0.8)
#Get the date at that position
second_split_date = second_dates[second_split_index]

#Compares split_date with model_df dates and determines if True or False
second_train_mask = second_model_df["date"] < second_split_date
second_test_mask = second_model_df["date"] >= second_split_date

#Selects only rows where train and test mask is true for X_train and X_test
second_X_train = second_X.loc[second_train_mask]
second_X_test = second_X.loc[second_test_mask]

#Selects only rows where train and test mask is true for y_train and y_test
second_y_train = second_y.loc[second_train_mask]
second_y_test = second_y.loc[second_test_mask]

print("Split date:", second_split_date)

print("X_train:", second_X_train.shape)
print("X_test:", second_X_test.shape)

print("y_train:", second_y_train.shape)
print("y_test:", second_y_test.shape)

Split date: 2026-08-03 00:00:00
X_train: (14400, 80)
X_test: (3600, 80)
y_train: (14400,)
y_test: (3600,)


In [18]:
#Tune Hyperparams
second_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    objective="reg:squarederror",
    random_state=42)

In [19]:
#Train
second_model.fit(second_X_train, second_y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [20]:
#Test
second_predictions = second_model.predict(second_X_test)

In [21]:
#Evaluate
second_mae = mean_absolute_error(second_y_test, second_predictions)
second_rmse = root_mean_squared_error(second_y_test, second_predictions)
second_r2 = r2_score(second_y_test, second_predictions)

print(f"MAE: {second_mae:.2f}")
print(f"RMSE: {second_rmse:.2f}")
print(f"R²: {second_r2:.4f}")

MAE: 5.93
RMSE: 12.86
R²: 0.8938


# Model Evaluation: Compare XGBoost Against Linear Regression

In [22]:
#LINEAR REGRESSION: Tomorrow's Forecast

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

In [23]:
#Evaluate
lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = root_mean_squared_error(y_test, lr_predictions)
lr_r2 = r2_score(y_test, lr_predictions)

print(f"MAE: {lr_mae:.2f}")
print(f"RMSE: {lr_rmse:.2f}")
print(f"R²: {lr_r2:.4f}")

MAE: 4.66
RMSE: 11.99
R²: 0.9126


In [24]:
#LINEAR REGRESSION: Average for the next seven days

second_lr_model = LinearRegression()

second_lr_model.fit(second_X_train, second_y_train)

second_lr_predictions = second_lr_model.predict(second_X_test)

In [25]:
#Evaluate
second_lr_mae = mean_absolute_error(second_y_test, second_lr_predictions)
second_lr_rmse = root_mean_squared_error(second_y_test, second_lr_predictions)
second_lr_r2 = r2_score(second_y_test, second_lr_predictions)

print(f"MAE: {second_lr_mae:.2f}")
print(f"RMSE: {second_lr_rmse:.2f}")
print(f"R²: {second_lr_r2:.4f}")

MAE: 6.02
RMSE: 12.52
R²: 0.8993


In [26]:
results = pd.DataFrame({
    "Model": ["Linear Regression","XGBoost","Linear Regression","XGBoost"],
    "Forecast": ["Tomorrow","Tomorrow","Next 7 Days","Next 7 Days"],
    "MAE": [lr_mae,mae,second_lr_mae,second_mae],
    "RMSE": [lr_rmse,rmse,second_lr_rmse,second_rmse],
    "R2": [lr_r2,r2,second_lr_r2,second_r2]})

In [27]:
display(results)

,Model,Forecast,MAE,RMSE,R2
0,Linear Regression,Tomorrow,4.663365,11.990603,0.912618
1,XGBoost,Tomorrow,5.107762,13.288911,0.892671
2,Linear Regression,Next 7 Days,6.017329,12.522611,0.899288
3,XGBoost,Next 7 Days,5.929123,12.857801,0.893825


# Save Trained Models

In [28]:
#Save the weights to the folder
folderloc = Path("drive/MyDrive/model")
folderloc.mkdir(parents=True, exist_ok=True)


model.save_model(folderloc / "xgboost_model_forecast_day.json")
second_model.save_model(folderloc / "xgboost_model_forecast_week.json")

joblib.dump(lr_model, folderloc / "lr_model_forecast_day.pkl")
joblib.dump(second_lr_model, folderloc / "lr_model_forecast_week.pkl")
joblib.dump(X_train.columns.tolist(), folderloc / "lr_day_features.pkl")
joblib.dump(second_X_train.columns.tolist(), folderloc / "lr_week_features.pkl")

['drive/MyDrive/model/lr_week_features.pkl']